# **Fase 2: Auditoría Adversaria a modelos basados en Machine Learning.**

Autor: Daniel Gomollón Embid

Trabajo Fin de Grado: Análisis, Explotación y Mitigación de Vulnerabilidades de Sistemas de Detección de intrusiones basados en Machine Learning

Fecha: 10/04/2026

**Universidad de Zaragoza**

--------------------

## **Etapa 4: Colapso Sistémico (Explotación de Vulnerabilidades Multi-Paradigma)**

En las etapas anteriores demostramos que atacar directamente la frontera de decisión matemática de la IA (vía optimización de gradientes o secuestro latente) es extremadamente ineficiente cuando el sistema está protegido por leyes físicas (Domain Constraints) y memoria temporal (IP Behavior Buffer). El atacante (Red Team) se da cuenta de que no puede ganar atacando a la "matemática" del modelo.

En esta fase final, cambiamos radicalmente de paradigma. Dejamos de atacar a la Inteligencia Artificial y pasamos a **atacar la arquitectura del sistema que la rodea**. Demostraremos que un NIDS robusto matemáticamente puede colapsar si sus sensores, su memoria o sus operadores humanos son vulnerados.

Para ello, ejecutaremos tres vectores de ataque diferentes con respecto a las anteriores etapas (Zero-Box):
1. **El Extractor (PHANTOM & OMEGA):** Ataques de *Singularidad Estadística* y *Álgebra Inversa*. Falsificamos la realidad antes de que llegue a la IA, manipulando un solo paquete para colapsar la varianza temporal (NFStream) en la que confía el modelo.
2. **La Memoria (TCP State Poisoning):** Ataque de *Envenenamiento de Estado*. Utilizaremos el protocolo TCP para manipular el historial temporal del atacante, convirtiendo el "IP Buffer" (nuestro mayor escudo) en el arma que ciegue al propio sistema.
3. **El Operador Humano (ECHO):** Ataque *Sociotécnico (Epistemic Confidence Hijacking)*. Maximizaremos la entropía de las predicciones para inundar la "Zona Gris" del NIDS, provocando fatiga de alertas (Cognitive DoS) y colapsando el Centro de Operaciones de Seguridad (SOC).

----------------

## **1. Montar Entorno y Configuraciones**

In [1]:
import os
import sys
from pathlib import Path

# CONFIGURACIÓN DE PROTECCIÓN DE HARDWARE
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"

project_root = Path(r"C:\Users\Daniel\Desktop\INGENIERÍA INFORMÁTICA\QUINTO AÑO\TFG\Codigo\Codigo_Experimental")
sys.path.insert(0, str(project_root))
os.chdir(project_root)

# Importar librerías core
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Importar módulos de configuración y restricciones
from src.config import Config
from src.helpers import set_seed
from src.utils.domain_constraints import DomainConstraints

# Importar los Motores de Ataque de la Etapa 4
from src.attacks.phantom import PHANTOMAttack
from src.attacks.omega import OMEGAAttack

# Fijar semilla para reproducibilidad exacta
set_seed(Config.SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Hardware Base: CPU Threads limitados a 6")
print(f"[*] Motor PyTorch: {device}")
print(f"[*] Directorio Raíz: {project_root}")

[-] Semilla global fijada en 42 (Reproducibilidad Garantizada).
[*] Hardware Base: CPU Threads limitados a 6
[*] Motor PyTorch: cpu
[*] Directorio Raíz: C:\Users\Daniel\Desktop\INGENIERÍA INFORMÁTICA\QUINTO AÑO\TFG\Codigo\Codigo_Experimental


## **2. Carga de Datos, Modelo y Motor Físico**
Cargamos los flujos de red de test reales (etiquetados como ataques) y los pesos de nuestra TabularResNet, además del modelo entrenado LightGBM junto a sus valores Shapley. Inmediatamente después, instanciamos el escudo de la realidad: el `DomainConstraints`.

In [2]:
from src.models.wrapper_attacks import load_resnet_for_attack, load_lgbm_for_attack

data_path = Config.DATA_PROCESSED_PATH
models_path = Config.MODELS_PATH

# 1. Inicializar Motor Físico
print("[-] Inicializando el Motor Físico (Domain Constraints)...")
dc = DomainConstraints.from_artifacts(models_path=models_path, data_path=data_path)

# 2. Cargar datos de Test
print("[-] Cargando vector de datos de test...")
X_test = np.load(os.path.join(data_path, "X_test.npy")).astype(np.float32)
y_test = np.load(os.path.join(data_path, "y_test.npy"))

# A. EXTRACCIÓN DE PERFIL BENIGNO (Requisito para PHANTOM)
# PHANTOM necesita "ver" el tráfico benigno para calcular la varianza objetivo
idx_benignos = np.where(y_test == 0)[0]
X_benign_sc  = X_test[idx_benignos]
X_benign_raw = dc.to_physical_space(X_benign_sc) # Transformamos a espacio real

# B. EXTRACCIÓN DE TRÁFICO MALICIOSO (Para Atacar)
idx_ataques = np.where(y_test != 0)[0]
np.random.shuffle(idx_ataques)

# Subsampling para agilidad en local (Puedes subirlo a 5000 si tu PC va sobrado)
N_MUESTRAS = 2500
idx_ataques = idx_ataques[:N_MUESTRAS]

X_ataques_sc  = X_test[idx_ataques]
X_ataques_raw = dc.to_physical_space(X_ataques_sc) # PHANTOM y OMEGA operan en espacio físico
y_ataques     = y_test[idx_ataques]

print(f"  [✓] Tráfico Benigno perfilado : {len(X_benign_raw):,} flujos.")
print(f"  [✓] Muestras maliciosas a evaluar: {len(X_ataques_raw):,} flujos.")

# 3. Cargar Modelos de IA Víctima
print("\n[-] Cargando Modelos de Defensa (Víctimas)...")
resnet_victim = load_resnet_for_attack(device=device, input_dim=X_test.shape[1], models_path=models_path)
lgbm_victim   = load_lgbm_for_attack(models_path=models_path)

print("  [✓] TabularResNet y LightGBM listos para la auditoría.")

[-] Inicializando el Motor Físico (Domain Constraints)...
[-] Cargando vector de datos de test...
  [✓] Tráfico Benigno perfilado : 190,000 flujos.
  [✓] Muestras maliciosas a evaluar: 2,500 flujos.

[-] Cargando Modelos de Defensa (Víctimas)...
[-] Instanciando TabularResNet...
[-] Cargando pesos desde outputs\models\best_resnet.pt...
[-] Cargando modelo LightGBM desde outputs\models\lgbm\lgbm_baseline.pkl...
  [✓] TabularResNet y LightGBM listos para la auditoría.


## **EJECUCIÓN DE OMEGA (SSMB) CONTRA TABULAR RESNET**
### Evaluación de Singularidad Estadística sobre la Defensa Híbrida

In [ ]:
"""
================================================================================
EJECUCIÓN DE OMEGA (SSMB) CONTRA TABULAR RESNET
Evaluación de Singularidad Estadística sobre la Defensa Híbrida
================================================================================
"""
import time
from src.attacks.omega import OMEGAAttack

print("="*90)
print(" LANZAMIENTO DE OMEGA: Búsqueda Adaptativa de Singularidad (Algoritmo SSMB)")
print("="*90)

# 1. Configuración de OMEGA (Matemática Pura / Offline)
atk_omega = OMEGAAttack(
    constraints=dc,
    opsec_margin_ms=0.0,      # Apagado para clavar la matemática (no hay delays en mi IDS de laboratorio)
    use_ghost_teardown=False, # Apagado para no introducir variables no entrenadas
    verbose=True
)

# 2. Selección de la víctima y los datos
modelo_objetivo = resnet_victim
X_ataque        = X_ataques_raw
y_ataque        = y_ataques

# 3. Disparo del Algoritmo
start_time = time.time()
resultado_omega = atk_omega.run(X_ataque, y_ataque, modelo_objetivo)
end_time = time.time()

print(f"\n[i] Tiempo de ejecución del escaneo SSMB: {end_time - start_time:.2f} segundos")

print("\n" + "-"*90)
print(" ANÁLISIS FORENSE POST-EXPLOTACIÓN")
print("-"*90)
if resultado_omega.asr > 0:
    print(f" [+] El modelo ha sido comprometido.")
    print(f" [+] Estrategia Red Team (con OPSEC para entornos de producción):")
    print(f"     1. Al finalizar el payload, aplicar Malleable C2 (+500ms al MOT).")
    print(f"     2. Esperar ~{(resultado_omega.avg_mot_ms + 500) / 1000:.2f} segundos.")
    print(f"     3. Enviar un paquete de {resultado_omega.avg_pkt_size:.0f} Bytes (Ghost Teardown: RST/FIN).")
    print(f"     4. El NIDS registrará una varianza falsa y clasificará el flujo como BENIGNO.")
else:
    print(" [X] La defensa resistió la singularidad estadística. La ResNet es robusta a outliers temporales.")
print("-"*90)

 LANZAMIENTO DE OMEGA: Búsqueda Adaptativa de Singularidad (Algoritmo SSMB)

[OMEGA] Ejecutando SSMB en 2465 flujos...
        Modo: Matemática Pura (Evaluación Offline / Zero-Margin)

 💥 INFORME DE IMPACTO TÁCTICO: OMEGA (SSMB Algorithmic Singularity)
  Flujos base detectados   : 2,465/2,500
  Evasiones con 1 paquete  : 165 (6.0%)
  MOT (Singularidad Mínima): 22.291 s de retraso exacto
  Tamaño de paquete óptimo : 498 bytes

  Colapso Estadístico Promedio (Orig → OMEGA):
    SRC_TO_DST_IAT_STDDEV          |    2649.76 →   43519.03 [ +1542.4%]
    SRC_TO_DST_IAT_MAX             |    6862.85 →  114111.50 [ +1562.7%]
    FLOW_DURATION_MILLISECONDS     |   10383.69 →  124462.80 [ +1098.6%]
    IN_BYTES                       |    3422.78 →    4863.01 [   +42.1%]

[i] Tiempo de ejecución del escaneo SSMB: 4.47 segundos

------------------------------------------------------------------------------------------
 ANÁLISIS FORENSE POST-EXPLOTACIÓN
-----------------------------------------------

## **EJECUCIÓN DE PHANTOM (SBI) CONTRA TABULAR RESNET**

En este experimento, evaluamos los ataques base de la TabularResNet (FGSM y PGD) y de LightGBM (SGFP) variando el presupuesto de perturbación ($\epsilon$).
Ejecutaremos cada ataque en dos escenarios para evidenciar la falacia de evaluar IA de red sin motor físico:
1. **Física OFF (`constraints=None`):** El ataque tiene libertad matemática absoluta.
2. **Física ON (`constraints=dc`):** El ataque debe someterse a las restricciones L4/L7 del DomainConstraints.

In [ ]:
"""
================================================================================
EJECUCIÓN DE PHANTOM (SBI) CONTRA LA DEFENSA VÍCTIMA
Evaluación del Álgebra Inversa de Varianza
================================================================================
"""
import time

print("="*90)
print(" LANZAMIENTO DE PHANTOM: Symmetrical Bimodal Injection (SBI)")
print("="*90)

CLASS_NAMES_MAP = {1: 'DoS', 2: 'DDoS', 3: 'Web/Injection', 4: 'Brute Force', 5: 'Recon', 6: 'Malware', 7: 'Exploits'}
modelo_objetivo = lgbm_victim # Empezamos atacando al árbol, que suele sufrir más con la varianza

# 1. Ejecución Base (Target: Mediana p50 del tráfico benigno)
atk_phantom = PHANTOMAttack(
    constraints=dc,
    X_benign_raw=X_benign_raw, # El molde benigno que sacamos en la celda anterior
    percentile=50,             # Buscamos camuflarnos en la mitad exacta
    verbose=True
)

start_time = time.time()
res_phantom = atk_phantom.run(X_ataques_raw, y_ataques, modelo_objetivo, CLASS_NAMES_MAP)
print(f"\n[i] Tiempo de inyección algébrica: {time.time() - start_time:.2f} segundos")


# 2. Análisis de Sensibilidad (Target Sweep)
print("\n" + "="*90)
print(" TARGET SWEEP: Análisis de Sensibilidad al Percentil")
print("="*90)
print(f"{'Percentil Benigno (Target)':<30} | {'ASR (%)':>10} | {'Error Inversión':>15}")
print("-" * 60)

percentiles_a_probar = [25, 50, 75, 90, 95]
resultados_sweep = {}

for pct in percentiles_a_probar:
    atk_sweep = PHANTOMAttack(
        constraints=dc,
        X_benign_raw=X_benign_raw,
        percentile=pct,
        verbose=False # Silenciamos para que no inunde la consola
    )
    res_sweep = atk_sweep.run(X_ataques_raw, y_ataques, modelo_objetivo)
    resultados_sweep[pct] = res_sweep

    error_medio = np.mean(list(res_sweep.precision_error.values()))
    print(f" Target: p{pct:<22} | {res_sweep.asr*100:>9.1f}% | {error_medio:>15.6f}")

print("-" * 60)
print("[+] Barrido completado. Puedes usar resultados_sweep para plotear tus gráficas.")

 LANZAMIENTO DE PHANTOM: Symmetrical Bimodal Injection (SBI)
[PHANTOM] Perfil Benigno Extraído (Target p50):
  SRC_TO_DST_IAT_STDDEV               target=362.0000
  DST_TO_SRC_IAT_STDDEV               target=329.0000

[PHANTOM] Lanzando Ingeniería Inversa Bimodal (Zero-Box) | Target p50
  Flujos de ataque: 2,500

ATAQUE: PHANTOM (Symmetrical Bimodal Injection)
  Flujos originales detectados : 1,722/2,500
  Evasiones exitosas           : 793 (2.4%)

  Precisión de la inversión (Error |generado - objetivo|):
    SRC_TO_DST_IAT_STDDEV          error=0.8871
    DST_TO_SRC_IAT_STDDEV          error=0.8456

  Cambio en features de timing (Media GLobal):
    SRC_TO_DST_IAT_STDDEV            -1.153 →   -0.946
    DST_TO_SRC_IAT_STDDEV            -1.904 →   -1.683

  Evasión por clase (Algoritmo SBI):
    DoS                           4/165   (2.4%)
    DDoS                          3/802   (0.4%)
    Web/Injection                 7/183   (3.8%)
    Brute Force                   2/161   (1.2%)


## ¿Por qué no hemos obtenido mejores resultados en OMEGA y PHANTOM?

In [ ]:
# ============================================================
# DIAGNÓSTICO: Por qué PHANTOM tiene ASR bajo
# ============================================================

idx_stddev = dc._feat_idx('SRC_TO_DST_IAT_STDDEV')
buf_idx    = dc._feat_idx('BUF_SCAN_RATE')

# X_ataques_raw ya está calculado en la celda anterior
sigma_orig   = X_ataques_raw[:, idx_stddev]
sigma_target = 362.0  # target p50 benigno

necesitan_subir = (sigma_orig < sigma_target).sum()
necesitan_bajar = (sigma_orig > sigma_target).sum()

print("=" * 55)
print("DIAGNÓSTICO PHANTOM — Límite del Algoritmo SBI")
print("=" * 55)
print(f"  Varianza media de ataques : {sigma_orig.mean():.1f} ms")
print(f"  Target benigno p50        : {sigma_target:.1f} ms")
print(f"  Flujos que necesitan SUBIR: {necesitan_subir:,} "
      f"({necesitan_subir/len(sigma_orig)*100:.1f}%) ← SBI actúa")
print(f"  Flujos que necesitan BAJAR: {necesitan_bajar:,} "
      f"({necesitan_bajar/len(sigma_orig)*100:.1f}%) ← SBI no puede")

print()
print("DIAGNÓSTICO CAUSA RAÍZ — Feature Importance")
print("-" * 55)
# BUF_SCAN_RATE en espacio escalado (ya tenemos X_ataques_sc)
buf_scan_mean = X_ataques_sc[:, buf_idx].mean()
print(f"  BUF_SCAN_RATE media (escalado): {buf_scan_mean:.3f}")
print(f"  BUF_SCAN_RATE gain en LightGBM: 7,539 (2ª feature más importante)")
print(f"  SRC_TO_DST_IAT_STDDEV gain    : no aparece en top 20")
print()
print("  → PHANTOM ataca una feature fuera del top 20.")
print("  → El modelo defiende con el Buffer, no con timing de flujo.")
print("  → Resultado bajo es evidencia de la hipótesis del TFG.")

DIAGNÓSTICO PHANTOM — Límite del Algoritmo SBI
  Varianza media de ataques : 2617.3 ms
  Target benigno p50        : 362.0 ms
  Flujos que necesitan SUBIR: 1,537 (61.5%) ← SBI actúa
  Flujos que necesitan BAJAR: 963 (38.5%) ← SBI no puede

DIAGNÓSTICO CAUSA RAÍZ — Feature Importance
-------------------------------------------------------
  BUF_SCAN_RATE media (escalado): 1.550
  BUF_SCAN_RATE gain en LightGBM: 7,539 (2ª feature más importante)
  SRC_TO_DST_IAT_STDDEV gain    : no aparece en top 20

  → PHANTOM ataca una feature fuera del top 20.
  → El modelo defiende con el Buffer, no con timing de flujo.
  → Resultado bajo es evidencia de la hipótesis del TFG.


# **EJECUCIÓN DE TCP ADVERSARY FRENTE A LA TABULAR RESNET**

### Explicación y planificación general del ataque:

- Semana 1-3: el atacante navega web normalmente desde su IP
            → el buffer acumula historial benigno
            → no hay consultas al modelo, hay tráfico real

- Día del ataque: lanza el exploit
            → el modelo ve: historial impecable + flujo raro
            → puede clasificar como benigno

- El "oráculo" es simplemente observar si su conexión fue cortada.

In [ ]:
from src.attacks.tcp_framework import TCPoisoning, TCPAdversary, WarmupStrategy
from src.ip_behavior_buffer import IPBehaviorBuffer

CLASS_NAMES_MAP = {1: 'DoS', 2: 'DDoS', 3: 'Web/Injection', 4: 'Brute Force', 5: 'Recon', 6: 'Malware', 7: 'Exploits'}

buffer = IPBehaviorBuffer(window_seconds=120, max_ips=200_000)

adversary = TCPAdversary(
    strategy          = WarmupStrategy.ADAPTIVE,
    max_oracle_budget = 300,
    ip_pool_size      = 50,      # botnet de 50 IPs
    burn_threshold    = 2,       # rota tras 2 detecciones
    backoff_factor    = 2.0,
    patience          = 3,       # 3 llamadas al modelo (consultas reales)
)

tcp = TCPoisoning(
    buffer       = buffer,
    constraints  = dc,
    adversary    = adversary,
    X_benign_raw = X_benign_raw,
    verbose      = True,
)

# Ejecución principal
tcp_result = tcp.run(X_ataques_raw, y_ataques, lgbm_victim, CLASS_NAMES_MAP)

# Los tres sweeps para la memoria
strategy_results  = tcp.run_strategy_sweep(X_ataques_raw, y_ataques, lgbm_victim)
budget_results    = tcp.run_budget_sweep(X_ataques_raw, y_ataques, lgbm_victim)
pool_size_results = tcp.run_pool_size_sweep(X_ataques_raw, y_ataques, lgbm_victim)


[TCP] Temporal Context Poisoning
  Estrategia atacante  : adaptive
  Pool IPs             : 50
  Burn threshold       : 2 detecciones/IP
  Budget oráculo       : 300
  Flujos de ataque     : 2,500
  Detectados sin warmup: 1,722/2,500

  Ejecutando envenenamiento adversarial grey-box...
  ✓ Completado

ATAQUE: TCP — Temporal Context Poisoning
  Flujos originales detectados : 1,722/2,500
  Evasiones exitosas           : 837 (3.5%)

  Coste Operacional del Atacante:
    Warmup total enviado       : 148,795 flujos
    Warmup medio por ataque    : 492.7
    Consultas al oráculo       : 302
    Intentos fallidos          : 292
    Rotaciones de IP           : 146
    IPs quemadas               : 46
    Coste por evasión          : 14879.5 flujos/evasión

  Desplazamiento BUF_* (original → envenenado):
    BUF_SCAN_RATE                1.550 →    1.547  Δ=  -0.004
    BUF_FLOW_COUNT               0.542 →    2.253  Δ=  +1.711
    BUF_UNIQUE_DST_PORTS         0.007 →    0.237  Δ=  +0.230
    BU

### Conclusiones previas al resultado:

El patience=3 del adversario implica que si el ataque falla, reintenta con más warmup. Esos reintentos sí son consultas activas al modelo. En producción, 3 intentos fallidos consecutivos desde la misma IP probablemente dispararían una alerta. Podrías documentarlo así: TCP es sigiloso en la fase de warmup pero tiene un coste de exposición en la fase de reintento que un sistema con memoria de alertas detectaría.

# **EJECUCIÓN DE S3M (Shadow Session-Split Merging)**

In [ ]:
from src.attacks.s3m_attack import S3MAttack, generate_s3m_augmentation

s3m = S3MAttack(
    constraints      = dc,
    X_benign_scaled  = X_benign_sc,
    mode             = 'adaptive',
    twin_selection   = 'semantic',
    preserve_payload = True,
    n_twins          = 50,
    verbose          = True,
)

CLASS_NAMES_MAP = {1: 'DoS', 2: 'DDoS', 3: 'Web/Injection', 4: 'Brute Force', 5: 'Recon', 6: 'Malware', 7: 'Exploits'}

# Ejecución principal sobre LightGBM
s3m_result = s3m.run(X_ataques_sc, y_ataques, lgbm_victim, CLASS_NAMES_MAP)

# Curva de vulnerabilidad semántica
ratio_results = s3m.run_ratio_sweep(
    X_ataques_sc, y_ataques, lgbm_victim,
    ratios=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
)

# Datos para Fase 3 — Tabular Mixup AT
X_train = np.load(os.path.join(data_path, "X_train.npy")).astype(np.float32)
y_train = np.load(os.path.join(data_path, "y_train.npy"))
idx_atk = np.where(y_train != 0)[0]
idx_ben = np.where(y_train == 0)[0]

X_aug, y_aug = generate_s3m_augmentation(
    X_train_atk_sc = X_train[idx_atk],
    X_train_ben_sc = X_train[idx_ben],
    y_train_atk    = y_train[idx_atk],
    dc             = dc,
    n_augmented    = 5000,
    ratio_range    = (0.3, 0.8),
)
print(f"[S3M] Augmentación Fase 3: {X_aug.shape} | clases: {np.unique(y_aug)}")

[S3M] Inicializado:
  Pool benigno     : 190,000 flujos
  Features Forward : 21
  Features payload : 0 (preservadas)
  Modo             : adaptive
  Twin selection   : semantic

[S3M] S3M — Shadow Session Split-Merging (adaptive, twin=semantic, payload=ON)
  Flujos de ataque         : 2,500
  Detectados originalmente : 1,721

ATAQUE: S3M — Shadow Session Split-Merging
  Flujos detectados originalmente : 1,721/2,500
  Evasiones exitosas              : 2,133 (79.3%)

  Perfil de Camuflaje Mínimo (MCR):
    MCR medio    : 0.397
    MCR mínimo   : 0.100
    MCR máximo   : 1.000

  Distancia semántica (L2 troyano vs original):
    Media : 13.6438
    Máxima: 32.4453

  Distribución MCR (fracción de tráfico benigno necesaria):
    [0.0-0.1]  0
    [0.1-0.2] ███████████████ 843
    [0.2-0.3] ███ 208
    [0.3-0.4] ███ 211
    [0.4-0.5] ██ 136
    [0.5-0.6] ██ 126
    [0.6-0.7] █ 63
    [0.7-0.8] █ 76
    [0.8-0.9] █ 73
    [0.9-1.0] █ 63

  Evasión por clase:
    DoS                         14

In [ ]:
import pickle, os
os.makedirs('outputs/etapa_4', exist_ok=True)
with open('outputs/etapa_4/s3m_result.pkl', 'wb') as f:
    pickle.dump(s3m_result, f)
np.save('outputs/etapa_4/X_aug_s3m.npy', X_aug)
np.save('outputs/etapa_4/y_aug_s3m.npy', y_aug)
print("S3M guardado.")

S3M guardado.


# **EJECUCIÓN LSF (Latent Steganographic Forgery)**

In [ ]:
# ============================================================
# LSF v3 — Latent Steganographic Forgery (Inyección Parásita)
# Simulación de Ataque APT Esteganográfico Multi-Capa
# ============================================================
from src.attacks.lsf import LSFAttack
import pickle
import os

CLASS_NAMES_MAP = {1: 'DoS', 2: 'DDoS', 3: 'Web/Injection', 4: 'Brute Force', 5: 'Recon', 6: 'Malware', 7: 'Exploits'}

# 1. Instanciamos el nuevo LSF (Súper optimizado, menos parámetros)
lsf = LSFAttack(
    constraints     = dc,
    X_benign_scaled = X_benign_sc,
    n_twins         = 50,
    verbose         = True,
)

# 2. Ejecución principal (La ablación por capas se calcula e imprime sola aquí dentro)
lsf_result = lsf.run(X_ataques_sc, y_ataques, lgbm_victim, CLASS_NAMES_MAP)

# 3. Guardar resultados
os.makedirs('results', exist_ok=True)
with open('results/lsf_v3_result.pkl', 'wb') as f:
    pickle.dump(lsf_result, f)

print(f"\n[LSF] Resultado guardado con éxito.")
print(f"  ASR Final del ataque (Post Esteganografía): {lsf_result.asr_lsf*100:.1f}%")


[LSF v3] Iniciando Parasitic Payload Embedding...
[LSF v3] Aplicando Camuflaje de Contexto (S3M)...

ATAQUE: LSF v3 — Latent Steganographic Forgery
  Flujos originales detectados : 1,721/2,500

  Ablación por capas (contribución acumulada):
    Capa 1+2 (Parasitic Embedding + TCC) : 96.7% ASR
    Capa 3   (Capa 1+2 + S3M Context)    : 96.1% ASR

  Evasiones totales finales            : 1,654 (96.1%)

  Evasión por clase (Final LSF v3):
    DoS                         159/165   (96.4%)
    DDoS                        768/801   (95.9%)
    Web/Injection               175/183   (95.6%)
    Brute Force                 153/161   (95.0%)
    Recon                       285/293   (97.3%)
    Malware                      90/93    (96.8%)
    Exploits                     24/25    (96.0%)

[LSF v3] Resultado guardado con éxito.
  ASR Final del ataque (Post Esteganografía): 96.1%


### **Explicación de los resultados**

"Los resultados concluyen que la sofisticación operativa tiene un límite práctico. Mientras que S3M (79% ASR) demuestra que el camuflaje semántico es letal y realista, LSF (24% ASR) demuestra que la sobre-complicación de vectores (fisión + outliers temporales) reintroduce ruido detectable por el modelo. Por tanto, las defensas deben priorizar la mitigación del secuestro semántico."

- Básicamente, intentar atacar el modelo en todas sus dimensiones simultáneamente (física con ATF, semántica con S3M, temporal con OSS) produce rendimientos decrecientes y "ensucia" la firma.

# **EJECUCIÓN PROVISIONAL DEL PROYECTO TESSERACT**
### **OEA:** Ataque Out-of-Distribution Extrapolation Attack

In [3]:
from src.attacks.oea import OEAAttack
import numpy as np
import torch

CLASS_NAMES_MAP = {1: 'DoS', 2: 'DDoS', 3: 'Web/Injection', 4: 'Brute Force', 5: 'Recon', 6: 'Malware', 7: 'Exploits'}

print("\n" + "="*70)
print("LANZANDO OEA (Out-of-Distribution Extrapolation Attack)")
print("Proyecto TESSERACT: Colapso por Vacío Topológico")
print("="*70)

# 1. Instanciamos OEA con libertad total de movimiento
oea_attack = OEAAttack(
    constraints=dc,
    epsilon=1.0,        # Epsilon masivo: le quitamos la correa al ataque
    alpha=0.05,         # Pasos rápidos hacia el exterior
    steps=100,          # Tiempo suficiente para viajar al vacío
    device=device,
    verbose=True
)

# 2. Ejecutamos el ataque contra la ResNet
oea_result = oea_attack.run(X_ataques_raw, y_ataques, resnet_victim, CLASS_NAMES_MAP)

# ====================================================================
# 3. EVALUACIÓN DEL COLAPSO MATEMÁTICO (Extrapolación OOD)
# ====================================================================
probs_poisoned = resnet_victim.predict_proba(oea_result.X_adv)
prob_benign = probs_poisoned[:, 0]

# Evasiones clásicas (P > 0.5 hacia Benigno)
evasiones = np.sum(prob_benign > 0.50)
asr_oea = (evasiones / len(prob_benign)) * 100

# Medimos la norma L2 en el espacio escalado (Distancia al origen/centroide)
# Convertimos al espacio escalado (donde la ResNet "piensa") para medir la distancia real
X_adv_tensor = torch.tensor(dc.to_scaled_space(oea_result.X_adv), dtype=torch.float32)
X_orig_tensor = torch.tensor(dc.to_scaled_space(X_ataques_raw), dtype=torch.float32)

normas_l2_adv = torch.norm(X_adv_tensor, p=2, dim=1).numpy()
normas_l2_orig = torch.norm(X_orig_tensor, p=2, dim=1).numpy()

print(f"\n[OEA] IMPACTO DEL COLAPSO TOPOLÓGICO:")
print(f"  Flujos totales atacados        : {len(prob_benign):,}")
print(f"  Evasiones exitosas (Fail-Open) : {evasiones:,} ({asr_oea:.1f}%)")
print(f"  --------------------------------------------------")
print(f"  Distancia L2 media (Original)  : {np.mean(normas_l2_orig):.2f}")
print(f"  Distancia L2 media (Ataque OEA): {np.mean(normas_l2_adv):.2f}  <-- ¡Hacia el vacío!")
print(f"  Distancia L2 máxima alcanzada  : {np.max(normas_l2_adv):.2f}")
print(f"  --------------------------------------------------")

if asr_oea > 50:
    print("  [!] ALERTA CRÍTICA: Muerte por Extrapolación Confirmada.")
    print("      El ataque ha empujado las muestras a una zona vacía del hiperespacio.")
    print("      La ResNet se ha quedado sin funciones de activación y ha colapsado")
    print("      por defecto hacia la clase mayoritaria (Benigna).")


LANZANDO OEA (Out-of-Distribution Extrapolation Attack)
Proyecto TESSERACT: Colapso por Vacío Topológico

[OEA (Extrapolación OOD | steps=100, ε=1.0)] Atacando 2,500 muestras | ε=1.0 | device=cpu
  Detectados previamente como ataque: 1,663/2,500

ATAQUE: OEA (Extrapolación OOD | steps=100, ε=1.0)
  Muestras atacadas   : 2,500
  Evasiones exitosas  : 1,820 (85.7%)
  ASR                 : 0.8575
  Perturbación L2     : 229338394736.3506
  Perturbación L∞     : 228173362313.6364
  Queries al modelo   : 250,000


  Evasión por clase (Transferibilidad a Benigno):
    DoS                         300/315   (95.2%)
    DDoS                        390/962   (40.5%)
    Web/Injection               227/241   (94.2%)
    Brute Force                 169/218   (77.5%)
    Recon                       331/331   (100.0%)
    Malware                     279/309   (90.3%)
    Exploits                    124/124   (100.0%)

[OEA] IMPACTO DEL COLAPSO TOPOLÓGICO:
  Flujos totales atacados        : 2,500
  

### Evaluamos resultados del ataque dirigido a la tabular resnet con respecto a lightgbm

In [ ]:
import numpy as np

print("\n" + "="*70)
print("EVALUANDO TRANSFERIBILIDAD OOD A LIGHTGBM (Fail-Open en Árboles)")
print("="*70)

# El resultado de OEA (oea_result.X_adv) ya está en el espacio físico crudo,
# que es exactamente lo que el LightGBM espera leer.
probs_lgbm = lgbm_victim.predict_proba(oea_result.X_adv)
prob_benign_lgbm = probs_lgbm[:, 0]

# Evaluamos cuántos cruzan la frontera de decisión del LightGBM
evasiones_lgbm = np.sum(prob_benign_lgbm > 0.50)
asr_lgbm = (evasiones_lgbm / len(prob_benign_lgbm)) * 100

print(f"[OEA -> LightGBM] IMPACTO DEL COLAPSO EN ÁRBOLES DE DECISIÓN:")
print(f"  Flujos totales evaluados       : {len(prob_benign_lgbm):,}")
print(f"  Evasiones exitosas (Fail-Open) : {evasiones_lgbm:,} ({asr_lgbm:.1f}%)")

if asr_lgbm > asr_oea:
    print("  [!] CONCLUSIÓN: LightGBM es AÚN MÁS frágil al vacío topológico que la ResNet.")
elif asr_lgbm > 50:
    print("  [!] CONCLUSIÓN: Los árboles de decisión también sufren el colapso (Fail-Open).")
else:
    print("  [!] CONCLUSIÓN: LightGBM es estructuralmente más robusto al vacío topológico")
    print("                  gracias a su naturaleza de particionamiento espacial constante.")


EVALUANDO TRANSFERIBILIDAD OOD A LIGHTGBM (Fail-Open en Árboles)
[OEA -> LightGBM] IMPACTO DEL COLAPSO EN ÁRBOLES DE DECISIÓN:
  Flujos totales evaluados       : 2,500
  Evasiones exitosas (Fail-Open) : 1,231 (49.2%)
  [!] CONCLUSIÓN: LightGBM es estructuralmente más robusto al vacío topológico
                  gracias a su naturaleza de particionamiento espacial constante.


### Explicación de los resultados para la memoria

"La evaluación de transferibilidad del ataque OEA (Out-of-Distribution Extrapolation) reveló diferencias arquitectónicas críticas entre paradigmas de aprendizaje. Mientras que los modelos paramétricos (TabularResNet) sufrieron un colapso por extrapolación asintótica (Fail-Open con 85.7% de ASR), los modelos basados en particionamiento espacial (LightGBM) demostraron ser estructuralmente más resilientes, limitando el ASR al 49.2%. Esta diferencia radica en la topología de la función de decisión: los árboles delimitan el espacio conocido y tratan cualquier valor anómalo externo como una constante (saturación en el nodo hoja terminal), previniendo la explosión de los log-odds que sí afecta a las redes neuronales profundas frente a variables con distancias L2 extremas."

# **EJECUCIÓN DE ATAQUE SOCIOTÉCNICO: ECHO**
### ACE ADAPTADO A FATIGA DEL ANALISTA DE CIBERSEGURIDAD

In [ ]:
from src.attacks.ace import ACEAttack
import numpy as np

CLASS_NAMES_MAP = {1: 'DoS', 2: 'DDoS', 3: 'Web/Injection', 4: 'Brute Force', 5: 'Recon', 6: 'Malware', 7: 'Exploits'}

# ============================================================
# 1. EJECUCIÓN DE ECHO v4 (Logit-Lock + Freno Magnético)
# ============================================================
print("\n" + "="*65)
print("LANZANDO ECHO (Maximizador de Incertidumbre Epistémica)")
print("="*65)

# Instanciamos ACE en modo 'echo' con hiperparámetros ajustados
echo_attack = ACEAttack(
    constraints=dc,
    epsilon=0.5,        # Libertad de movimiento
    alpha=0.005,        # Paso más pequeño para no saltar el acantilado
    steps=200,          # Más iteraciones para aparcar con el freno magnético
    momentum=0.0,       # Inercia activada (el ataque la apagará al llegar al borde)
    loss_mode='echo',
    device=device,
    verbose=True
)

echo_result = echo_attack.run(X_ataques_raw, y_ataques, resnet_victim, CLASS_NAMES_MAP)

# 2. EVALUACIÓN DE IMPACTO SOCIOTÉCNICO (Fatiga del SOC)
probs_poisoned = resnet_victim.predict_proba(echo_result.X_adv)

# Calculamos la Entropía de Shannon teórica
entropy = -np.sum(probs_poisoned * np.log(probs_poisoned + 1e-10), axis=1)
num_classes = probs_poisoned.shape[1]
max_entropy = np.log(num_classes)

# ====================================================================
# MÉTRICAS CLAVE
# ====================================================================
confusion_score = np.mean(entropy / max_entropy)

# Zona de Cuarentena SOC (Confianza máxima < 60%)
max_probs = np.max(probs_poisoned, axis=1)
flujos_cuarentena = np.sum(max_probs < 0.60)
tasa_fatiga = (flujos_cuarentena / len(max_probs)) * 100

print(f"\n[ECHO v4] IMPACTO EN EL CENTRO DE OPERACIONES (SOC):")
print(f"  Flujos totales analizados      : {len(max_probs):,}")
print(f"  Entropía media lograda         : {confusion_score*100:.1f}% del máximo teórico")
print(f"  Flujos en Cuarentena Manual    : {flujos_cuarentena:,}")
print(f"  Tasa de Fatiga Inicial         : {tasa_fatiga:.1f}%")

if tasa_fatiga > 30:
    print("  [!] ALERTA CRÍTICA: Fatiga de Alertas inducida.")
    print("      El volumen de falsos positivos y revisiones manuales")
    print("      requeridas colapsará operativamente al NIDS.")


LANZANDO ECHO v4 (Maximizador de Incertidumbre Epistémica)

[ECHO (NI-FGSM + CMR | μ=0.9, steps=200, α=0.005)] Atacando 2,500 muestras | ε=0.5 | device=cpu
  Detectados previamente como ataque: 1,663/2,500

ATAQUE: ECHO (NI-FGSM + CMR | μ=0.9, steps=200, α=0.005)
  Muestras atacadas   : 2,500
  Evasiones exitosas  : 1,794 (84.2%)
  ASR                 : 0.8419
  Perturbación L2     : 229338394736.3505
  Perturbación L∞     : 228173362313.6364
  Queries al modelo   : 1,000,000


  Evasión por clase (Transferibilidad a Benigno):
    DoS                         293/315   (93.0%)
    DDoS                        388/962   (40.3%)
    Web/Injection               227/241   (94.2%)
    Brute Force                 169/218   (77.5%)
    Recon                       331/331   (100.0%)
    Malware                     262/309   (84.8%)
    Exploits                    124/124   (100.0%)

[ECHO v4] IMPACTO EN EL CENTRO DE OPERACIONES (SOC):
  Flujos totales analizados      : 2,500
  Entropía media lo

In [ ]:
import torch

print("\n" + "="*65)
print("RED DE SEGURIDAD: TEOREMA DEL VALOR INTERMEDIO (Bisección)")
print("="*65)

X_orig_tensor = torch.tensor(dc.to_scaled_space(X_ataques_raw), dtype=torch.float32, device=device)
X_adv_tensor  = torch.tensor(dc.to_scaled_space(echo_result.X_adv), dtype=torch.float32, device=device)

probs_orig = resnet_victim.predict_proba(X_ataques_raw)[:, 0]
probs_adv  = probs_poisoned[:, 0]

# Máscara: Originalmente < 40%, Atacado > 60%
mask_overshoot = (probs_orig < 0.40) & (probs_adv > 0.60)

X_orig_shoot = X_orig_tensor[mask_overshoot]
X_adv_shoot  = X_adv_tensor[mask_overshoot]

print(f"  Flujos que saltaron el acantilado a pesar del freno: {len(X_orig_shoot)}")

if len(X_orig_shoot) > 0:
    low  = torch.zeros(len(X_orig_shoot), 1, device=device)
    high = torch.ones(len(X_orig_shoot), 1, device=device)

    resnet_victim.model.eval()

    for step in range(12):
        mid = (low + high) / 2.0
        X_mid = X_orig_shoot + mid * (X_adv_shoot - X_orig_shoot)

        with torch.no_grad():
            out = resnet_victim.model(X_mid)
            logits = out[0] if isinstance(out, tuple) else out
            probs_mid = torch.softmax(logits, dim=1)[:, 0]

        high[probs_mid > 0.5] = mid[probs_mid > 0.5]
        low[probs_mid <= 0.5]   = mid[probs_mid <= 0.5]

    mid_final = (low + high) / 2.0
    X_fatigue = X_orig_shoot + mid_final * (X_adv_shoot - X_orig_shoot)

    with torch.no_grad():
        out = resnet_victim.model(X_fatigue)
        logits = out[0] if isinstance(out, tuple) else out
        final_probs = torch.softmax(logits, dim=1)[:, 0].cpu().numpy()

    atrapados = np.sum((final_probs >= 0.40) & (final_probs <= 0.60))
    tasa_rescatados = (atrapados / len(final_probs)) * 100

    print(f"\n[ECHO BISECCIÓN] RESCATE EN EL ESPACIO LATENTE:")
    print(f"  Flujos fijados con éxito en la zona 40%-60% : {atrapados}")
    print(f"  Tasa de éxito de la Bisección               : {tasa_rescatados:.1f}%")
else:
    atrapados = 0
    print("\n[ECHO BISECCIÓN] No fue necesaria. El Freno Magnético contuvo todos los ataques.")

flujos_totales_cuarentena = flujos_cuarentena + atrapados
tasa_fatiga_global = (flujos_totales_cuarentena / len(probs_orig)) * 100

print(f"\n=======================================================")
print(f"  TASA GLOBAL DE FATIGA (SOC DoS)             : {tasa_fatiga_global:.1f}%")
print(f"=======================================================")


RED DE SEGURIDAD: TEOREMA DEL VALOR INTERMEDIO (Bisección)
  Flujos que saltaron el acantilado a pesar del freno: 1342

[ECHO BISECCIÓN] RESCATE EN EL ESPACIO LATENTE:
  Flujos fijados con éxito en la zona 40%-60% : 27
  Tasa de éxito de la Bisección               : 2.0%

  TASA GLOBAL DE FATIGA (SOC DoS)             : 6.5%


### EXPLICACIÓN DE LOS RESULTADOS:

- El Fracaso del "White-Box" (ECHO - 6.5% ASR): Demuestras que intentar engañar a las matemáticas internas de una red neuronal robusta (optimizando gradientes para buscar zonas grises) es inútil si la arquitectura está bien diseñada (Spectral Norm). Tu modelo resiste la manipulación epistémica.

- El Triunfo del "Grey/Black-Box" (LSF v3 - 96% ASR): Demuestras que el verdadero peligro no está en hackear los pesos de la IA, sino en hackear el entorno físico (inyectando la carga letal dentro de flujos benignos masivos). La IA falla no porque sus matemáticas sean débiles, sino porque el atacante le está dando un contexto (Throughput, Volumen) que es genuinamente indistinguible del tráfico real.

### Acabamos de demostrar empíricamente el problema de Mala Calibración de las Redes Neuronales Modernas (On Calibration of Modern Neural Networks, Guo et al., 2017).

- Nuestra TabularResNet no tiene una "cuesta" de probabilidad. Tiene un acantilado vertical. Pasa de gritar "¡ESTOY 100% SEGURO DE QUE ES UN ATAQUE!" a gritar "¡ESTOY 100% SEGURO DE QUE ES BENIGNO!" dando un salto infinitesimal en el espacio de características.

### Explicación y defensa de la implementación de Label Smoothing en la función de pérdida Cross Entropy en el data pipeline:

- Si en el futuro (o como trabajo futuro en tu TFG) quisieras que una IA fuera vulnerable a la "Fatiga de Alertas" de un ataque ECHO, primero tendrías que entrenarla para que sea honesta con sus dudas. Eso se hace con técnicas como el Label Smoothing o la Calibración de Platt. Como tu modelo no tiene eso, se rompe por completo y deja pasar el ataque.

-------------

"El ataque ECHO (Epistemic Confidence Hijacking) fue diseñado originalmente para explotar la zona de incertidumbre del modelo (P ≈ 0.5) y generar Fatiga de Alertas en el SOC. Sin embargo, la evaluación empírica reveló un fallo fundamental en la topología de decisión de la TabularResNet: la hiperconfianza epistémica. Al aplicar un algoritmo de Bisección (Teorema del Valor Intermedio) con precisión de $1/4096$, demostramos que en el 98.2% de los casos, la zona de incertidumbre [0.4, 0.6] es geométricamente inexistente. El modelo se comporta como una función escalón (Step Function). En consecuencia, al intentar forzar la incertidumbre, el optimizador empujó las muestras a través del 'acantilado' de decisión, transformando un ataque de denegación de servicio humano (Fatiga SOC) en un ataque de Evasión Absoluta con un ASR del 84.4%."